# Rounding-convention tests — Q2 / Q7 / Q8 (integer-exact)

Scope: the algo-table **rounding convention** items.

* **Q2** — mantissa rounding: **truncate toward zero** (truncate magnitude,
  preserve sign"), HW-confirmed. No code change was needed: the divisor is an
  exact power of two and `.int()` truncates toward zero (`-0.3 -> 0`, never
  -1). The quantizer is bit-exact as written; this file locks that in with
  regression asserts.
* **Q7** — PCU output rounding: **round half up** (ties toward `+inf` for both
  signs: `+2.5 -> +3`, `-1.5 -> -1`), implemented integer-exact in `pcu_div`.
  If silicon ever turns out to mean *half away from zero* (`-1.5 -> -2`), only
  the negative-tie rows change.
* **Q8** — bias rescale into accumulator units is done **offline** (SW choice):
  **round half up** (`floor(v + 0.5)`, ties toward `+inf`, matching the PCU),
  error <= 0.5 ULP; the bias enters the pipe as `int64` so `pcu_pla -> pcu_div`
  stays integer end-to-end.

Run all cells top to bottom; every assertion passing = tests pass.


In [1]:
import os
import sys
# make npe_mx_emu importable whether jupyter runs from the repo root or from EL_nb/
_here = os.getcwd()
_cands = [_here, os.path.dirname(_here)]  # EL_nb/ -> parent
for _c in _cands:
    if os.path.isdir(os.path.join(_c, 'npe_mx_emu')) and _c not in sys.path:
        sys.path.insert(0, _c)
        break

import torch

from npe_mx_emu import (
    Cstruct,
    LayerMXQParams,
    base_mx_quantizer,
    build_c_struct_arr,
    pcu,
    pcu_div,
)

def ref_half_up(x, k):
    """Pure-python oracle for round half up (ties toward +inf)."""
    return (x + (1 << (k - 1))) >> k if k > 0 else (x << (-k))

print("imports ok")


imports ok


## Q2 — mantissa rounding: truncate toward zero (truncate magnitude, preserve sign)

`base_mx_quantizer` divides by an exact power of two and casts with `.int()`:
positive truncation drops the tail, negative truncation drops the magnitude
and keeps the sign, sub-LSB negatives go to 0 (e.g. `-0.3 -> 0`, never -1).
(Floor era, Q1: the block-max mantissa now fills `[64, 127]`.)
Synced with HW (algo table Q2); these asserts lock it in.


In [2]:

# ---- Q2 / Test: quantizer truncation vs oracle -------------------------------
# .int() semantics: truncate toward zero (the load-bearing cast)
sem = torch.tensor([3.9, -3.9, -0.3]).int()
assert sem.tolist() == [3, -3, 0], "Q2: .int() must truncate toward zero"

# case 1: mixed signs in one block -> [124, -124, -9, 64] (floor era: E=7, divisor 2^1)
CAL = 6
x1 = torch.tensor([3.9, -3.9, -0.3, 2.0]) * 2.0 ** CAL
mx1 = base_mx_quantizer(x1, block_size=4, calibrate_offset=CAL)
want1 = [int(v) for v in x1 / 2]  # oracle: trunc toward zero
assert mx1.data.flatten().tolist() == want1 == [124, -124, -9, 64], f"Q2: {mx1.data.flatten().tolist()}"
print("Q2 case1:", mx1.data.flatten().tolist(), "(shared_exp", int(mx1.shared_exp.flatten()[0]), "= E - m_off)")

# Q1 (floor) property: the block-max mantissa fills [64, 127] -- the integer bit is used
amax_mant = int(mx1.data.abs().max())
assert 64 <= amax_mant <= 127, f"Q1: block-max mantissa {amax_mant} outside [64,127]"
print("Q1 property: block-max mantissa", amax_mant, "in [64, 127]")

# case 2: sub-LSB negative truncates to 0, not -1 -> [100, 0] (floor era: E=12, divisor 2^6)
x2 = torch.tensor([100.0, -0.3]) * 2.0 ** CAL
mx2 = base_mx_quantizer(x2, block_size=2, calibrate_offset=CAL)
assert mx2.data.flatten().tolist() == [100, 0], f"Q2: {mx2.data.flatten().tolist()}"
amax2 = int(mx2.data.abs().max())
assert 64 <= amax2 <= 127, f"Q1: block-max mantissa {amax2} outside [64,127]"
print("Q2 case2:", mx2.data.flatten().tolist(), "(-0.3 -> 0, sign preserved)")

print("Q2 OK: truncate toward zero, magnitude dropped, sign preserved")


Q2 case1: [124, -124, -9, 64] (shared_exp 1 = E - m_off)
Q1 property: block-max mantissa 124 in [64, 127]
Q2 case2: [100, 0] (-0.3 -> 0, sign preserved)
Q2 OK: truncate toward zero, magnitude dropped, sign preserved


## Q7 — exhaustive agreement with the oracle + tie table

Sweep `x in [-2000, 2000]` x `k in [1..8]` on int64 tensors: `pcu_div` in
`"round"` mode must equal the pure-python half-up oracle everywhere, and
`"floor"` must equal `x >> k` (its pre-Q7 behavior, regression check).
Then the explicit tie table: every exact-half case, both signs.


In [3]:
# ---- Q7 / Test A: exhaustive oracle sweep ------------------------------------
xs = torch.arange(-2000, 2001, dtype=torch.int64)
for k in range(1, 9):
    got  = pcu_div(xs, shift_right=-k, round_type="round")
    want = torch.tensor([ref_half_up(int(x), k) for x in xs], dtype=torch.int64)
    assert torch.equal(got, want), f"Q7: round-half-up != oracle at k={k}"

    got_f = pcu_div(xs, shift_right=-k, round_type="floor")
    assert torch.equal(got_f, xs >> k), f"Q7: floor regression at k={k}"

print("Q7 Test A OK: round==oracle and floor==x>>k for 4001 x 8 cases")


Q7 Test A OK: round==oracle and floor==x>>k for 4001 x 8 cases


In [4]:
# ---- Q7 / Test B: explicit tie table (k=2 -> divide by 4) -------------------
# x : exact value x/4 : expected round-half-up
tie_table = {
     2:  1,   # +0.5 -> +1
     6:  2,   # +1.5 -> +2
    10:  3,   # +2.5 -> +3
    14:  4,   # +3.5 -> +4
    -2:  0,   # -0.5 ->  0   (up, not away from zero)
    -6: -1,   # -1.5 -> -1
   -10: -2,   # -2.5 -> -2
    # non-ties nearby
     1:  0,   3:  1,   5:  1,   7:  2,   9:  2,  11:  3,
    -1:  0,  -3: -1,  -5: -1,  -7: -2,  -9: -2, -11: -3,
}
xs = torch.tensor(list(tie_table.keys()), dtype=torch.int64)
got = pcu_div(xs, shift_right=-2, round_type="round")
for x, want, g in zip(tie_table.keys(), tie_table.values(), got):
    assert int(g) == want, f"Q7: tie/non-tie mismatch at x={x}: got {int(g)}, want {want}"
    print(f"x={x:>4}  x/4={x/4:+.2f}  -> {int(g):>3}")
print("Q7 Test B OK: ties go toward +inf for both signs")


x=   2  x/4=+0.50  ->   1
x=   6  x/4=+1.50  ->   2
x=  10  x/4=+2.50  ->   3
x=  14  x/4=+3.50  ->   4
x=  -2  x/4=-0.50  ->   0
x=  -6  x/4=-1.50  ->  -1
x= -10  x/4=-2.50  ->  -2
x=   1  x/4=+0.25  ->   0
x=   3  x/4=+0.75  ->   1
x=   5  x/4=+1.25  ->   1
x=   7  x/4=+1.75  ->   2
x=   9  x/4=+2.25  ->   2
x=  11  x/4=+2.75  ->   3
x=  -1  x/4=-0.25  ->   0
x=  -3  x/4=-0.75  ->  -1
x=  -5  x/4=-1.25  ->  -1
x=  -7  x/4=-1.75  ->  -2
x=  -9  x/4=-2.25  ->  -2
x= -11  x/4=-2.75  ->  -3
Q7 Test B OK: ties go toward +inf for both signs


In [5]:
# ---- Q7 / Test C: tie beyond the 2^53 float64 range --------------------------
k = 4
big = (((1 << 53) + 2) << k) | (1 << (k - 1))   # even quotient, exact tie bit, x ~ 2^57
x = torch.tensor([big], dtype=torch.int64)

got_int   = int(pcu_div(x, shift_right=-k, round_type="round")[0])
want      = ref_half_up(big, k)
# old float path: float64 cannot even represent big (tie bit lost) and gets it wrong
float_val = int((torch.tensor([float(big)]) * 2.0 ** (-k)).round()[0].item())

print(f"big           : {big}  (float64-exact: {float(big) == big})")
print(f"int pcu_div   : {got_int}  (rounds up: tie detected)")
print(f"oracle        : {want}")
print(f"old float path: {float_val}  (misses the tie)")
assert got_int == want, "Q7: integer path lost a tie beyond 2^53"
assert float_val != want, "sanity: this case must break the float path"
print("Q7 Test C OK: integer path detects ties where float64 cannot")


big           : 144115188075855912  (float64-exact: False)
int pcu_div   : 9007199254740995  (rounds up: tie detected)
oracle        : 9007199254740995
old float path: 9007199254740992  (misses the tie)
Q7 Test C OK: integer path detects ties where float64 cannot


In [6]:
# ---- Q7 / Test D: trunc mode, left-shift branch, pcu() integration ----------
# trunc: toward zero for both signs (changed vs old de-facto floor on negatives)
xs = torch.tensor([-14, -13, -12, -11, -2, -1, 0, 1, 2, 11, 13], dtype=torch.int64)
want_trunc = torch.tensor([int(x / 4) for x in xs], dtype=torch.int64)  # int() truncates
assert torch.equal(pcu_div(xs, shift_right=-2, round_type="trunc"), want_trunc), "Q7: trunc"
print("trunc check:", want_trunc.tolist())

# shift_right >= 0: exact left shift for every mode
for mode in ("floor", "round", "trunc"):
    assert torch.equal(pcu_div(xs, shift_right=3, round_type=mode), xs << 3), "Q7: left shift"
print("left-shift check ok")

# unknown mode raises
try:
    pcu_div(xs, shift_right=-2, round_type="ceil")
    raise AssertionError("Q7: unknown mode should have raised")
except KeyError:
    pass

# pcu() integration: p_sum -> pla(scale=1,bias=0) -> div(shift=-2, round) -> relux
c = Cstruct(bias=0, scale=1, shift=-2, round_type="round", min_val=-8, max_val=8)
p_sum = torch.tensor([13, -13, 2, -2, 100, -100], dtype=torch.int64)
# 13/4=3.25->3 ; -13/4=-3.25->-3 ; 2/4=0.5->1 (tie up) ; -2/4=-0.5->0 (tie up)
# 100/4=25 -> clamp 8 ; -100/4=-25 -> clamp -8
want = torch.tensor([3, -3, 1, 0, 8, -8])
assert torch.equal(pcu(p_sum, c, e_min_layer=0), want), "Q7: pcu integration"
print("pcu integration check:", want.tolist())

print("Q7 Test D OK: trunc, left shift, mode guard, pcu() integration")


trunc check: [-3, -3, -3, -2, 0, 0, 0, 0, 0, 2, 3]
left-shift check ok
pcu integration check: [3, -3, 1, 0, 8, -8]
Q7 Test D OK: trunc, left shift, mode guard, pcu() integration


## Q8 — bias rescale into accumulator units: offline, round half up

Bias lives in accumulator units `2^(x_cal + w_cal)`; the rescale is offline
(SW choice, algo table Q8): `floor(v + 0.5)` = ties toward `+inf`, error
<= 0.5 ULP, and the stored bias is `int64` so `pcu_pla -> pcu_div` never
leaves the integer domain (float promotion above 2^53 is impossible).


In [7]:
# ---- Q8 / Test: bias conversion oracle + dtype + real-path integration -------
CAL = 4  # x_cal + w_cal
qp = LayerMXQParams(x_cal=2, w_cal=2)
assert qp.b_cal == CAL

def ref_half_up_offline(v):
    import math
    return math.floor(v + 0.5)

# conversion table: ties (x.5 fractions) and non-ties, both signs
biases = [0.09375, -0.09375, 1.2, -1.2, 1.7, -1.7]
scaled = [b * 2.0 ** CAL for b in biases]
got = build_c_struct_arr(qp, torch.tensor(biases), len(biases))
for oc, (b, v) in enumerate(zip(biases, scaled)):
    b_int = got[oc].bias
    want = ref_half_up_offline(v)
    assert b_int.dtype == torch.int64, "Q8: bias must be int64 (integer pla->div path)"
    assert int(b_int) == want, f"Q8: bias {b} -> {int(b_int)}, want {want}"
    print(f"bias {b:+.5f} *2^{CAL} = {v:+.4f} -> {int(b_int):+d}  (old floor: {v // 1:+.0f} / {int(v // 1):+d})")

# real-path integration: build Cstruct via build_c_struct_arr, run pcu() on int64
c = build_c_struct_arr(qp, torch.tensor([0.5]), 1)[0]   # 0.5*2^4 = 8 -> bias 8
assert int(c.bias) == 8
p_sum = torch.tensor([4, -4, 2 ** 55], dtype=torch.int64)  # 2^55: beyond float64-exact range
# pla: p_sum + 8 ; div shift = 0 + e_min_layer = -2 -> half-up:
# (4+8)=12 -> 3 ; (-4+8)=4 -> 1 ; (2^55+8) -> 2^53+2 -> relux clamp 2^23-1
want = torch.tensor([3, 1, 2 ** 23 - 1])
out = pcu(p_sum, c, e_min_layer=-2)  # e_min_layer supplies the >>2 (build_c_struct keeps shift=0)
assert torch.equal(out, want), f"Q8: pcu integration {out.tolist()} != {want.tolist()}"
print("Q8: pcu() through build_c_struct_arr:", out.tolist(), "(int64 end-to-end, relux clamped)")

print("Q8 OK: bias round half up, int64 storage, integer pla->div path")


bias +0.09375 *2^4 = +1.5000 -> +2  (old floor: +1 / +1)
bias -0.09375 *2^4 = -1.5000 -> -1  (old floor: -2 / -2)
bias +1.20000 *2^4 = +19.2000 -> +19  (old floor: +19 / +19)
bias -1.20000 *2^4 = -19.2000 -> -19  (old floor: -20 / -20)
bias +1.70000 *2^4 = +27.2000 -> +27  (old floor: +27 / +27)
bias -1.70000 *2^4 = -27.2000 -> -27  (old floor: -28 / -28)
Q8: pcu() through build_c_struct_arr: [3, 1, 8388607] (int64 end-to-end, relux clamped)
Q8 OK: bias round half up, int64 storage, integer pla->div path


In [8]:
print("=" * 60)
print("ALL Q2 / Q7 / Q8 TESTS PASSED")
print("  Q2: mantissa truncate toward zero (magnitude dropped, sign preserved)")
print("  Q7: pcu_div round half up, integer-exact, ties toward +inf")
print("  Q8: bias offline, round half up, int64 (pla->div integer end-to-end)")


ALL Q2 / Q7 / Q8 TESTS PASSED
  Q2: mantissa truncate toward zero (magnitude dropped, sign preserved)
  Q7: pcu_div round half up, integer-exact, ties toward +inf
  Q8: bias offline, round half up, int64 (pla->div integer end-to-end)
